In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_PRIVATE_CREDIT")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_private_credit_links.parquet", index=False)

In [6]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 171


In [7]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\(\s*(\[\[.*?\]\])\s*\);'

sleep_duration = 5

In [8]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [9]:
# DataFrames de salida
dfs_quarters = []
dfs_months = []

start_time = time.time()
for url in country_links:
    country = urlparse(url).path.strip("/").split("/")[0]

    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            # --- separar según formato de Period ---
            if df["Period"].str.contains(r"Q\d").any():
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_quarters.append(df_pivot)
                print(f"✅ Datos trimestrales extraídos para {country}")
            else:
                df_pivot = df.pivot(index="Period", columns="Country", values="Value")
                dfs_months.append(df_pivot)
                print(f"✅ Datos mensuales extraídos para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

# unir cada grupo por separado
df_quarters = pd.concat(dfs_quarters, axis=1) if dfs_quarters else pd.DataFrame()
df_months = pd.concat(dfs_months, axis=1) if dfs_months else pd.DataFrame()

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Antigua-and-Barbuda no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
⏭️ País Aruba no está en la lista, se omite.
✅ Datos mensuales extraídos para Australia
✅ Datos trimestrales extraídos para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Bahrain no está en la lista, se omite.
⏭️ País Bangladesh no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos mensuales extraídos para Belgium
⏭️ País Belize no está en la lista, se omite.
⏭️ País Benin no está en la lista, se omite.
⏭️ País Bolivia no está en la lista, se omite.
⏭️ País Bosnia-and-Herzegovina no está en la lista, se omite.
⏭️ País Botswana no está en la lista, se om

In [10]:
# resultados finales
print("\n📊 DataFrame trimestral")
df_quarters.head()


📊 DataFrame trimestral


Country,Austria,Philippines
Period,,
Q1 1996,133.25,NaN
Q1 1997,142.70,NaN
Q1 1998,152.34,NaN
Q1 1999,160.72,NaN
Q1 2000,174.30,NaN


In [11]:
print("\n📊 DataFrame mensual")
df_months.head()


📊 DataFrame mensual


Country,Australia,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Slovenia,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom
Period,,,,,,,,,,,,,,,,,,,,,
Apr 2003,726.22,172.88,383.31,6.89,1667.40,28310.53,NaN,52938.82,NaN,NaN,...,NaN,704.13,742441.4,722.81,2296.37,615.12,5945.60,33.36,NaN,1041.44
Apr 2004,817.75,177.20,435.55,10.31,1789.41,30714.55,NaN,56949.67,NaN,NaN,...,NaN,774.57,802220.3,832.20,2405.36,626.31,6119.73,54.18,NaN,1120.06
Apr 2005,914.62,185.09,530.00,15.81,1931.69,35643.63,NaN,64904.64,NaN,NaN,...,NaN,930.21,827935.8,993.56,2599.54,651.10,6803.53,83.21,4320.47,1202.40
Apr 2006,1048.26,200.36,637.44,18.73,2092.91,42713.69,NaN,77927.20,NaN,25.94,...,NaN,1141.49,906815.4,1279.30,2922.13,680.07,7228.36,129.24,4816.09,1293.77
Apr 2007,1193.00,219.78,781.89,26.12,2363.23,49825.01,NaN,103444.80,NaN,29.81,...,20.89,1453.88,1049289.1,1576.90,3245.51,781.40,7515.50,176.78,5343.29,1409.61


In [12]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [13]:
# URL específica para Paraguay
response = requests.get(custom_url, headers=headers)
html = response.text

# Buscar el bloque JSON dentro del HTML
match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    # Crear DataFrame con los datos
    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = "Paraguay"  # Asignar explícitamente

    # Pivotear para formato ancho (opcional)
    df_paraguay = df.pivot(index="Period", columns="Country", values="Value")

    print("✅ Datos de Paraguay extraídos correctamente")
else:
    print("⚠️ No se encontraron datos en el HTML para Paraguay")

✅ Datos de Paraguay extraídos correctamente


In [14]:
df_paraguay.tail()

Country,Paraguay
Period,
Sep 2020,102957.00
Sep 2021,115220.50
Sep 2022,129783.89
Sep 2023,142860.38
Sep 2024,166684.36


In [15]:
# Unir ambos DataFrames por el índice 'Period', preservando todos los datos
new_month_df = pd.merge(df_months, df_paraguay, how="outer", left_index=True, right_index=True)
new_month_df.head()

Country,Australia,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Apr 1965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1966,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.77,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1967,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Apr 1969,NaN,NaN,NaN,NaN,76.62,NaN,NaN,NaN,NaN,NaN,...,4.76,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
def split_period_index(df):
    """
    Recibe un DataFrame con índice 'Period' (ej: 'Apr 1960' o 'Q1 1990')
    Devuelve el mismo DataFrame con columnas 'Year' y 'Month' o 'Quarter'
    """
    df = df.copy().reset_index()  # sacar Period del índice
    df["Year"] = None
    df["Month"] = None
    df["Quarter"] = None

    for i, period in enumerate(df["Period"]):
        parts = period.split()

        if len(parts) == 2:
            first, year = parts
            df.at[i, "Year"] = int(year)

            if "Q" in first:  # Formato trimestral
                df.at[i, "Quarter"] = first
            else:  # Formato mensual
                df.at[i, "Month"] = first

    return df.set_index("Period")  # opcional: devolver con Period otra vez de índice

In [17]:
def month_to_quarter(df):
    """
    Takes a DataFrame with 'Month' and 'Quarter' columns.
    If 'Month' exists, it assigns the corresponding quarter to 'Quarter'.
    """
    df = df.copy()

    # Dictionary for mapping months → quarters
    month_to_q = {
        "Jan": 1, "Feb": 1, "Mar": 1,
        "Apr": 2, "May": 2, "June": 2,
        "Jul": 3, "Aug": 3, "Sep": 3,
        "Oct": 4, "Nov": 4, "Dec": 4,
    }

    # Update quarter only where Month exists
    df["Quarter"] = df.apply(
        lambda row: month_to_q.get(row["Month"], row["Quarter"]),
        axis=1
    )

    return df

In [18]:
def quarter_to_int(df):
    """
    Convierte los valores de la columna 'Quarter' a enteros (Q1 → 1, Q2 → 2, etc.)
    """
    df = df.copy()

    q_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}

    df["Quarter"] = df["Quarter"].map(q_map).fillna(df["Quarter"])

    return df

In [19]:
def year_to_int(df):
    """
    Convierte la columna 'Year' a enteros.
    Si hay valores no numéricos, los convierte en NaN.
    """
    df = df.copy()
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
    return df

In [22]:
new_quarter_df = split_period_index(df_quarters)
new_quarter_df = quarter_to_int(new_quarter_df)
new_quarter_df = year_to_int(new_quarter_df)
new_quarter_df.dtypes

Country
Austria        float64
Philippines    float64
Year             Int64
Month           object
Quarter          int64
dtype: object

In [24]:
new_quarter_df.tail()

Country,Austria,Philippines,Year,Month,Quarter
Period,,,,,
Q4 2020,334.12,8503.83,2020,None,4
Q4 2021,360.17,8891.59,2021,None,4
Q4 2022,381.32,10265.97,2022,None,4
Q4 2023,379.59,11142.73,2023,None,4
Q4 2024,379.19,12551.07,2024,None,4


In [25]:
new_month_df = split_period_index(new_month_df)
new_month_df = month_to_quarter(new_month_df)
new_month_df = quarter_to_int(new_month_df)
new_month_df = year_to_int(new_month_df)
new_month_df.dtypes

Country
Australia         float64
Belgium           float64
Brazil            float64
Bulgaria          float64
Canada            float64
Chile             float64
China             float64
Colombia          float64
Croatia           float64
Cyprus            float64
Czech-Republic    float64
Denmark           float64
Estonia           float64
Finland           float64
France            float64
Germany           float64
Greece            float64
Hong-Kong         float64
Hungary           float64
Iceland           float64
India             float64
Indonesia         float64
Ireland           float64
Israel            float64
Italy             float64
Japan             float64
Latvia            float64
Lithuania         float64
Luxembourg        float64
Macedonia         float64
Malaysia          float64
Malta             float64
Mexico            float64
Morocco           float64
Netherlands       float64
New-Zealand       float64
Norway            float64
Peru              float64
Pola

In [27]:
new_month_df.tail()

Country,Australia,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,Cyprus,...,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay,Year,Month,Quarter
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,2656.79,337.54,3816.70,59.95,5227.54,182947.95,167170.64,491135.59,29.30,29.01,...,6815.09,1222.74,18795.54,2763.79,8935.90,1950.39,102957.00,2020,Sep,3.0
Sep 2021,2798.94,360.76,4438.82,64.34,5604.59,192524.37,186973.00,522911.25,29.94,27.07,...,7156.31,1273.02,19614.18,3123.79,8717.06,2000.95,115220.50,2021,Sep,3.0
Sep 2022,3045.09,392.50,5162.92,72.76,6056.00,216732.78,207689.24,612999.30,33.26,25.94,...,7806.13,1314.64,20315.16,5171.32,9704.41,2087.93,129783.89,2022,Sep,3.0
Sep 2023,3208.11,401.25,5614.67,80.19,6264.30,222143.82,229391.60,646816.89,35.82,23.04,...,8002.97,1336.51,20489.53,8176.98,10142.10,2097.03,142860.38,2023,Sep,3.0
Sep 2024,3382.14,406.20,6216.18,91.19,6442.77,226636.56,246962.72,652364.66,38.59,23.03,...,7968.71,1357.65,20531.97,11550.68,10304.61,2108.30,166684.36,2024,Sep,3.0


In [28]:
unique_values = new_month_df["Quarter"].unique()
print(unique_values)

[2. 3. 4. 1.]


In [29]:
# Guardar como archivo Parquet
new_month_df.to_parquet("../../data/month_private_credit_values.parquet", index=False)
new_quarter_df.to_parquet("../../data/quarter_private_credit_values.parquet", index=False)